# Notebook 06 — Single-Nucleus Preprocessing
## GSE184950 — Human substantia nigra snRNA-seq

### Biological objective

This notebook prepares the independent single-nucleus RNA-seq cohort **GSE184950** for cell-type-resolved validation of the metabolic transcriptomic signature discovered in GSE136666.

The primary question at this stage is **not yet differential expression**. It is:

> Can the GSE184950 nuclei be transformed into a rigorously quality-controlled, donor-aware AnnData object while preserving raw counts for later donor × cell-type pseudobulk analysis?

### Study design principles

1. **Nuclei are not biological replicates.**
   - The biological unit is the donor.
   - Differential testing will later use donor-level pseudobulk.

2. **Raw counts are preserved.**
   - Raw integer counts are stored in `adata.layers["counts"]`.
   - Normalized/log-transformed values are used only for visualization, clustering and exploratory cell annotation.

3. **Control vs PD is the primary validation contrast.**
   - Parkinson's disease dementia (PDD) is retained but treated as a secondary/progression group.

4. **Repeated samples from the same donor are not treated as independent donors.**
   - GSE184950 contains 34 GEO sample rows but 32 unique donor IDs.
   - Repeated sample rows are explicitly tracked.

5. **QC thresholds are data-driven and sample-aware.**
   - Rigid universal thresholds are avoided where possible.
   - Filtering decisions are exported for reproducibility.

6. **Cell-type labels remain provisional until verified.**
   - Published annotation from `GSE184950_add2.xlsx`, if available, is preferred.
   - Marker-based labels are treated as provisional and are never silently presented as ground truth.

### Expected input files

At minimum:
- `GSE184950_series_matrix.txt`

For expression preprocessing, also provide either:
- `GSE184950_RAW.tar`, or
- extracted GSM/sample folders containing 10x files:
  - `matrix.mtx[.gz]`
  - `barcodes.tsv[.gz]`
  - `features.tsv[.gz]` or `genes.tsv[.gz]`

Optional:
- `GSE184950_add2.xlsx` for published/additional annotation.

### Main outputs

- cleaned sample/donor metadata;
- per-sample QC tables;
- merged quality-controlled AnnData object;
- raw-count layer for pseudobulk;
- normalized/log1p data for visualization;
- HVGs, PCA, neighbors, UMAP, Leiden clusters;
- preliminary marker scores and optional provisional cell-type labels;
- donor × disease × sample composition tables;
- manifest for Notebook 07.

In [1]:
# 0. Environment and reproducibility
from pathlib import Path
import csv
import json
import re
import sys
import tarfile
import shutil
import platform
import importlib.util
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

HAS_SCANPY = importlib.util.find_spec("scanpy") is not None
HAS_ANNDATA = importlib.util.find_spec("anndata") is not None
HAS_SCRUBLET = importlib.util.find_spec("scrublet") is not None

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy installed:", HAS_SCANPY)
print("anndata installed:", HAS_ANNDATA)
print("scrublet installed:", HAS_SCRUBLET)

Python: 3.13.5
Platform: Linux-6.18.44-x86_64-with-glibc2.41
pandas: 2.2.3
numpy: 2.3.5
scanpy installed: False
anndata installed: False
scrublet installed: False


## Recommended environment

Install the single-cell packages before running the full notebook:

```bash
pip install scanpy anndata scrublet openpyxl
```

A clean environment is preferable:

```bash
conda create -n parkinson_snRNA python=3.12
conda activate parkinson_snRNA
pip install jupyterlab scanpy anndata scrublet openpyxl
```

The notebook can still audit GEO metadata without Scanpy, but expression preprocessing requires Scanpy/AnnData.

In [2]:
# 1. Project structure and search paths
PROJECT_ROOT = Path.cwd()

SEARCH_ROOTS = [
    PROJECT_ROOT,
    PROJECT_ROOT.parent,
    PROJECT_ROOT / "data",
    PROJECT_ROOT.parent / "data",
    Path("/mnt/data"),
]

RAW_DIR = PROJECT_ROOT / "data" / "GSE184950" / "raw"
METADATA_DIR = PROJECT_ROOT / "data" / "GSE184950" / "metadata"
PROCESSED_DIR = PROJECT_ROOT / "data" / "GSE184950" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "single_cell"
FIGURES_DIR = PROJECT_ROOT / "figures" / "single_cell"
EXTRACT_DIR = PROJECT_ROOT / "data" / "GSE184950" / "extracted"

for d in [
    RAW_DIR, METADATA_DIR, PROCESSED_DIR,
    RESULTS_DIR, FIGURES_DIR, EXTRACT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

def find_files(pattern):
    matches = []
    for root in SEARCH_ROOTS:
        if root.exists():
            matches.extend(root.rglob(pattern))
    return sorted(set(p.resolve() for p in matches if p.is_file()))

def find_first(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]
    for pattern in patterns:
        hits = find_files(pattern)
        if hits:
            return hits[0]
    return None

SERIES_FILE = find_first([
    "GSE184950_series_matrix.txt",
    "*GSE184950*series_matrix*.txt",
])

RAW_TAR = find_first([
    "GSE184950_RAW.tar",
    "*GSE184950*RAW*.tar",
])

ADD2_FILE = find_first([
    "GSE184950_add2.xlsx",
    "*GSE184950*add2*.xlsx",
])

print("Series matrix:", SERIES_FILE)
print("RAW tar:", RAW_TAR)
print("Additional annotation:", ADD2_FILE)

assert SERIES_FILE is not None, (
    "GSE184950 series matrix was not found."
)

Series matrix: /mnt/data/GSE184950_series_matrix.txt
RAW tar: None
Additional annotation: None


# Part A — Parse GEO metadata

The GEO `series_matrix` contains sample-level metadata only.  
It does **not** contain the single-nucleus expression matrix.

The parser below extracts repeated `!Sample_characteristics_ch1` rows into structured columns.

In [3]:
# 2. Parse GEO series matrix
def parse_geo_series_matrix(path):
    path = Path(path)

    sample_records = {}
    sample_titles = None
    sample_accessions = None
    characteristics_rows = []
    supplementary_rows = []

    with path.open("r", encoding="utf-8", errors="replace") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if not line.startswith("!Sample_"):
                continue

            parts = next(csv.reader([line], delimiter="\t", quotechar='"'))
            key = parts[0]
            vals = parts[1:]

            if key == "!Sample_title":
                sample_titles = vals
            elif key == "!Sample_geo_accession":
                sample_accessions = vals
            elif key == "!Sample_characteristics_ch1":
                characteristics_rows.append(vals)
            elif key.startswith("!Sample_supplementary_file"):
                supplementary_rows.append((key, vals))
            else:
                sample_records.setdefault(key, vals)

    assert sample_titles is not None
    assert sample_accessions is not None
    assert len(sample_titles) == len(sample_accessions)

    n = len(sample_accessions)
    meta = pd.DataFrame({
        "sample_title": sample_titles,
        "geo_accession": sample_accessions,
    })

    # Parse characteristics: "key: value"
    for vals in characteristics_rows:
        if len(vals) != n:
            continue
        parsed = []
        keys = []
        for x in vals:
            if ":" in x:
                k, v = x.split(":", 1)
                keys.append(k.strip().lower())
                parsed.append(v.strip())
            else:
                keys.append("")
                parsed.append(x.strip())

        common_keys = pd.Series(keys).value_counts()
        if len(common_keys):
            col_key = common_keys.index[0]
            safe = re.sub(r"[^a-z0-9]+", "_", col_key).strip("_")
            meta[safe] = parsed

    # Attach supplementary URLs if available
    for key, vals in supplementary_rows:
        if len(vals) == n:
            safe = key.replace("!Sample_", "").lower()
            meta[safe] = vals

    return meta

meta = parse_geo_series_matrix(SERIES_FILE)

print("Sample rows:", len(meta))
print("Columns:", meta.columns.tolist())
display(meta.head())

Sample rows: 34
Columns: ['sample_title', 'geo_accession', 'tissue', 'disease_state', 'brain_bank_donor_id', 'age', 'gender', 'race', 'ethnicity', 'postmortem_interval_hours', 'braak_stage', 'supplementary_file_1']


,sample_title,geo_accession,tissue,disease_state,brain_bank_donor_id,age,gender,race,ethnicity,postmortem_interval_hours,braak_stage,supplementary_file_1
0,A10,GSM5602315,Substanstia nigra,Unaffected Control,HctZZT,84,Male,White,Not Hispanic,15.5,N/A,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...
1,A15,GSM5602316,Substanstia nigra,Parkinson's Disease Dementia,HBGK,74,Male,White,Hispanic or Latino,24.08,Stage II,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...
2,A18,GSM5602317,Substanstia nigra,Parkinson's Disease Dementia,HBBC,90,Male,White,Not Hispanic or Latino,21.3,Stage IV,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...
3,A2,GSM5602318,Substanstia nigra,Unaffected Control,HctYH,71,Female,White,Hispanic,16.4,N/A,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...
4,A29,GSM5602319,Substanstia nigra,Parkinson's Disease Dementia,HBOX,82,Male,White,Not Hispanic or Latino,4.2,Stage VI,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...


In [4]:
# 3. Standardize metadata fields
rename_candidates = {
    "disease_state": "disease_state",
    "brain_bank_donor_id": "donor_id",
    "age": "age",
    "gender": "sex",
    "postmortem_interval_hours": "PMI_hours",
    "braak_stage": "braak_stage",
    "tissue": "tissue",
}

for old, new in rename_candidates.items():
    if old in meta.columns and old != new:
        meta = meta.rename(columns={old: new})

# Disease labels
def standardize_disease(x):
    text = str(x).strip().lower()
    if "dementia" in text:
        return "PDD"
    if "parkinson" in text:
        return "PD"
    if "control" in text or "unaffected" in text:
        return "Control"
    return str(x)

meta["disease"] = meta["disease_state"].map(standardize_disease)

# Numeric fields
if "age" in meta.columns:
    meta["age"] = pd.to_numeric(meta["age"], errors="coerce")

if "PMI_hours" in meta.columns:
    meta["PMI_hours"] = pd.to_numeric(meta["PMI_hours"], errors="coerce")

# Braak numeric helper; N/A remains NaN
def braak_to_numeric(x):
    if pd.isna(x):
        return np.nan
    text = str(x).upper()
    roman = {
        "I": 1, "II": 2, "III": 3,
        "IV": 4, "V": 5, "VI": 6
    }
    for r, val in sorted(roman.items(), key=lambda z: -len(z[0])):
        if re.search(rf"\b{r}\b", text):
            return val
    return np.nan

if "braak_stage" in meta.columns:
    meta["braak_numeric"] = meta["braak_stage"].map(braak_to_numeric)

# Ensure strings
for c in ["geo_accession", "sample_title", "donor_id", "sex", "disease"]:
    if c in meta.columns:
        meta[c] = meta[c].astype(str)

display(meta.head())

,sample_title,geo_accession,tissue,disease_state,donor_id,age,sex,race,ethnicity,PMI_hours,braak_stage,supplementary_file_1,disease,braak_numeric
0,A10,GSM5602315,Substanstia nigra,Unaffected Control,HctZZT,84,Male,White,Not Hispanic,15.50,N/A,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...,Control,NaN
1,A15,GSM5602316,Substanstia nigra,Parkinson's Disease Dementia,HBGK,74,Male,White,Hispanic or Latino,24.08,Stage II,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...,PDD,2.0
2,A18,GSM5602317,Substanstia nigra,Parkinson's Disease Dementia,HBBC,90,Male,White,Not Hispanic or Latino,21.30,Stage IV,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...,PDD,4.0
3,A2,GSM5602318,Substanstia nigra,Unaffected Control,HctYH,71,Female,White,Hispanic,16.40,N/A,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...,Control,NaN
4,A29,GSM5602319,Substanstia nigra,Parkinson's Disease Dementia,HBOX,82,Male,White,Not Hispanic or Latino,4.20,Stage VI,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5602...,PDD,6.0


In [5]:
# 4. Study-design audit
study_audit = pd.Series({
    "sample_rows": len(meta),
    "unique_GSMs": meta["geo_accession"].nunique(),
    "unique_donors": meta["donor_id"].nunique(),
    "controls_samples": int((meta["disease"] == "Control").sum()),
    "PD_samples": int((meta["disease"] == "PD").sum()),
    "PDD_samples": int((meta["disease"] == "PDD").sum()),
})

display(study_audit)

print("\nSamples by disease:")
display(meta["disease"].value_counts().rename("sample_rows"))

print("\nUnique donors by disease:")
display(
    meta.groupby("disease")["donor_id"]
    .nunique()
    .rename("unique_donors")
)

sample_rows         34
unique_GSMs         34
unique_donors       32
controls_samples    10
PD_samples           6
PDD_samples         18
dtype: int64


Samples by disease:


disease
PDD        18
Control    10
PD          6
Name: sample_rows, dtype: int64


Unique donors by disease:


disease
Control     9
PD          6
PDD        17
Name: unique_donors, dtype: int64

In [6]:
# 5. Detect repeated donors
donor_sample_counts = (
    meta.groupby(["donor_id", "disease"])
    .agg(
        n_samples=("geo_accession", "nunique"),
        sample_titles=("sample_title", lambda x: ";".join(x)),
        GSMs=("geo_accession", lambda x: ";".join(x)),
    )
    .reset_index()
)

repeated_donors = donor_sample_counts[
    donor_sample_counts["n_samples"] > 1
].copy()

print("Repeated donors:", len(repeated_donors))
display(repeated_donors)

Repeated donors: 2


,donor_id,disease,n_samples,sample_titles,GSMs
9,HBHF,Control,2,p4;p5,GSM5602344;GSM5602345
16,HBNX,PDD,2,p7;p9,GSM5602346;GSM5602347


In [7]:
# 6. Covariate audit
covariate_cols = [
    c for c in [
        "disease", "age", "sex", "PMI_hours",
        "braak_stage", "braak_numeric", "race", "ethnicity"
    ] if c in meta.columns
]

covariate_missingness = (
    meta[covariate_cols]
    .isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

covariate_missingness["missing_fraction"] = (
    covariate_missingness["missing_values"] / len(meta)
)

display(covariate_missingness)

if {"disease", "sex"}.issubset(meta.columns):
    display(pd.crosstab(meta["disease"], meta["sex"]))

if {"disease", "age"}.issubset(meta.columns):
    display(
        meta.groupby("disease")["age"]
        .agg(["count", "mean", "median", "min", "max"])
    )

if {"disease", "PMI_hours"}.issubset(meta.columns):
    display(
        meta.groupby("disease")["PMI_hours"]
        .agg(["count", "mean", "median", "min", "max"])
    )

,missing_values,missing_fraction
disease,0,0.000000
age,0,0.000000
sex,0,0.000000
PMI_hours,0,0.000000
braak_stage,0,0.000000
braak_numeric,13,0.382353
race,0,0.000000
ethnicity,0,0.000000


sex,Female,Male
disease,,
Control,6,4
PD,2,4
PDD,4,14


,count,mean,median,min,max
disease,,,,,
Control,10,84.700000,86.5,71,93
PD,6,77.666667,76.0,69,91
PDD,18,79.166667,78.0,66,91


,count,mean,median,min,max
disease,,,,,
Control,10,17.897000,15.950,7.00,25.50
PD,6,16.205000,14.155,5.05,32.40
PDD,18,20.119444,20.660,4.20,30.05


In [8]:
# 7. Export cleaned metadata
meta.to_csv(
    METADATA_DIR / "GSE184950_clean_sample_metadata.csv",
    index=False,
)
repeated_donors.to_csv(
    METADATA_DIR / "GSE184950_repeated_donors.csv",
    index=False,
)
covariate_missingness.to_csv(
    RESULTS_DIR / "GSE184950_covariate_missingness.csv"
)

print("Metadata exports completed.")

Metadata exports completed.


# Part B — Locate / extract raw 10x matrices

GEO distributes this study as per-sample supplementary archives.  
The notebook supports either:

1. `GSE184950_RAW.tar` containing multiple GSM archives;
2. already-extracted `.tar.gz` sample archives;
3. already-extracted directories containing 10x files.

No expression analysis is attempted unless complete 10x triplets can be identified.

In [9]:
# 8. Extract GSE184950_RAW.tar if present
def safe_extract_tar(tar_path, destination):
    tar_path = Path(tar_path)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)

    with tarfile.open(tar_path, "r:*") as tar:
        destination_resolved = destination.resolve()
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise RuntimeError("Unsafe path detected in tar archive.")
        tar.extractall(destination)

if RAW_TAR is not None:
    raw_tar_extract = EXTRACT_DIR / "GSE184950_RAW"
    if not raw_tar_extract.exists() or not any(raw_tar_extract.iterdir()):
        print("Extracting:", RAW_TAR)
        safe_extract_tar(RAW_TAR, raw_tar_extract)
    else:
        print("RAW tar already extracted:", raw_tar_extract)
else:
    print("GSE184950_RAW.tar not present.")

GSE184950_RAW.tar not present.


In [10]:
# 9. Locate and extract per-sample archives
sample_archives = []

for root in SEARCH_ROOTS + [EXTRACT_DIR]:
    if root.exists():
        sample_archives.extend(root.rglob("GSM*.tar.gz"))
        sample_archives.extend(root.rglob("GSM*.tgz"))

sample_archives = sorted(set(p.resolve() for p in sample_archives))

print("Per-sample archives found:", len(sample_archives))

sample_extract_root = EXTRACT_DIR / "samples"
sample_extract_root.mkdir(parents=True, exist_ok=True)

for archive in sample_archives:
    m = re.search(r"(GSM\d+)", archive.name)
    gsm = m.group(1) if m else archive.stem
    dest = sample_extract_root / gsm

    if not dest.exists() or not any(dest.rglob("*")):
        dest.mkdir(parents=True, exist_ok=True)
        try:
            safe_extract_tar(archive, dest)
        except Exception as exc:
            warnings.warn(f"Could not extract {archive.name}: {exc}")

Per-sample archives found: 0


In [11]:
# 10. Detect valid 10x matrix directories
def is_10x_dir(folder):
    folder = Path(folder)
    names = {p.name for p in folder.iterdir()} if folder.is_dir() else set()

    has_matrix = any(
        n in names for n in ["matrix.mtx", "matrix.mtx.gz"]
    )
    has_barcodes = any(
        n in names for n in ["barcodes.tsv", "barcodes.tsv.gz"]
    )
    has_features = any(
        n in names for n in [
            "features.tsv", "features.tsv.gz",
            "genes.tsv", "genes.tsv.gz"
        ]
    )
    return has_matrix and has_barcodes and has_features

candidate_dirs = set()

for root in SEARCH_ROOTS + [EXTRACT_DIR]:
    if root.exists():
        for matrix_file in list(root.rglob("matrix.mtx")) + list(root.rglob("matrix.mtx.gz")):
            candidate_dirs.add(matrix_file.parent.resolve())

tenx_dirs = sorted(
    d for d in candidate_dirs if is_10x_dir(d)
)

def infer_gsm_from_path(path):
    text = str(path)
    m = re.search(r"(GSM\d+)", text)
    return m.group(1) if m else None

tenx_table = pd.DataFrame({
    "matrix_dir": [str(x) for x in tenx_dirs],
    "geo_accession": [infer_gsm_from_path(x) for x in tenx_dirs],
})

print("Valid 10x directories:", len(tenx_table))
display(tenx_table.head(20))

EXPRESSION_AVAILABLE = len(tenx_table) > 0
print("EXPRESSION_AVAILABLE =", EXPRESSION_AVAILABLE)

Valid 10x directories: 0


,matrix_dir,geo_accession


EXPRESSION_AVAILABLE = False


## Expression-data checkpoint

If `EXPRESSION_AVAILABLE = False`, stop interpretation here.

That means the GEO metadata were successfully audited, but the actual nucleus-by-gene matrices are not available locally yet.

The notebook deliberately **does not infer cell-level results from metadata alone**.

In [12]:
# 11. Optional published/additional annotation workbook audit
add2_sheets = {}

if ADD2_FILE is not None:
    try:
        xls = pd.ExcelFile(ADD2_FILE)
        print("Workbook sheets:", xls.sheet_names)

        for sheet in xls.sheet_names:
            df = pd.read_excel(ADD2_FILE, sheet_name=sheet)
            add2_sheets[sheet] = df
            print(f"{sheet}: {df.shape}")
            display(df.head())
    except Exception as exc:
        warnings.warn(f"Could not read GSE184950_add2.xlsx: {exc}")
else:
    print("GSE184950_add2.xlsx not present.")

GSE184950_add2.xlsx not present.


# Part C — Load and merge nuclei

This section requires Scanpy and complete 10x matrices.

Each sample is loaded independently, then:
- gene names are made unique;
- sample-level metadata are attached to every nucleus;
- barcode is preserved;
- cell IDs are made globally unique using the GSM/sample identifier.

Raw count values remain unchanged.

In [13]:
# 12. Import Scanpy/AnnData only if available
if HAS_SCANPY and HAS_ANNDATA:
    import scanpy as sc
    import anndata as ad
    from scipy import sparse

    sc.settings.verbosity = 2
    sc.settings.set_figure_params(dpi=90, facecolor="white")

    print("scanpy:", sc.__version__)
    print("anndata:", ad.__version__)
else:
    print(
        "Scanpy/AnnData unavailable. "
        "Install them before expression preprocessing."
    )

CAN_PROCESS_EXPRESSION = (
    EXPRESSION_AVAILABLE and HAS_SCANPY and HAS_ANNDATA
)
print("CAN_PROCESS_EXPRESSION =", CAN_PROCESS_EXPRESSION)

Scanpy/AnnData unavailable. Install them before expression preprocessing.
CAN_PROCESS_EXPRESSION = False


In [14]:
# 13. Read all detectable 10x samples
sample_adatas = []
load_log = []

if CAN_PROCESS_EXPRESSION:
    meta_by_gsm = meta.set_index("geo_accession")

    for _, row in tenx_table.iterrows():
        folder = Path(row["matrix_dir"])
        gsm = row["geo_accession"]

        if gsm is None:
            warnings.warn(
                f"Could not infer GSM from {folder}; sample skipped."
            )
            continue

        if gsm not in meta_by_gsm.index:
            warnings.warn(
                f"{gsm} is not represented in GEO metadata; skipped."
            )
            continue

        print("Loading", gsm, folder)

        try:
            a = sc.read_10x_mtx(
                folder,
                var_names="gene_symbols",
                make_unique=True,
                cache=False,
            )

            a.obs_names = [
                f"{gsm}_{bc}" for bc in a.obs_names.astype(str)
            ]

            a.obs["barcode_original"] = [
                x.split("_", 1)[1] for x in a.obs_names
            ]
            a.obs["geo_accession"] = gsm

            sample_meta = meta_by_gsm.loc[gsm]
            for c in meta.columns:
                if c == "geo_accession":
                    continue
                a.obs[c] = sample_meta[c]

            sample_adatas.append(a)
            load_log.append({
                "geo_accession": gsm,
                "status": "loaded",
                "n_nuclei": a.n_obs,
                "n_genes": a.n_vars,
                "matrix_dir": str(folder),
            })

        except Exception as exc:
            load_log.append({
                "geo_accession": gsm,
                "status": f"error: {exc}",
                "n_nuclei": np.nan,
                "n_genes": np.nan,
                "matrix_dir": str(folder),
            })

load_log = pd.DataFrame(load_log)
display(load_log)

if len(load_log):
    load_log.to_csv(
        RESULTS_DIR / "GSE184950_sample_load_log.csv",
        index=False,
    )

""


In [15]:
# 14. Concatenate samples
adata = None

if CAN_PROCESS_EXPRESSION and sample_adatas:
    adata = ad.concat(
        sample_adatas,
        join="outer",
        merge="same",
        label="loaded_sample_index",
        index_unique=None,
        fill_value=0,
    )

    adata.var_names_make_unique()

    print(adata)
    print("Nuclei:", adata.n_obs)
    print("Genes:", adata.n_vars)
    print("Samples:", adata.obs["geo_accession"].nunique())
    print("Donors:", adata.obs["donor_id"].nunique())

    assert adata.obs_names.is_unique
else:
    print("Merged AnnData not created.")

Merged AnnData not created.


# Part D — Raw-count preservation and QC metrics

For snRNA-seq, mitochondrial RNA often behaves differently from whole-cell scRNA-seq.  
We therefore calculate mitochondrial percentage but avoid applying an arbitrary universal cutoff before inspecting the data.

QC metrics:
- total UMI counts;
- detected genes;
- mitochondrial counts / percentage;
- optional ribosomal percentage;
- sample-specific distributions.

In [16]:
# 15. Preserve raw counts and calculate QC metrics
if adata is not None:
    adata.layers["counts"] = adata.X.copy()

    # Human gene-symbol conventions
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.upper().str.match(r"^RP[SL]")

    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo"],
        percent_top=None,
        log1p=False,
        inplace=True,
    )

    qc_cols = [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
    ]
    display(adata.obs[qc_cols].describe().T)

In [17]:
# 16. Per-sample QC summary
sample_qc_summary = pd.DataFrame()

if adata is not None:
    sample_qc_summary = (
        adata.obs
        .groupby(
            ["geo_accession", "sample_title", "donor_id", "disease"],
            observed=True
        )
        .agg(
            nuclei=("geo_accession", "size"),
            median_counts=("total_counts", "median"),
            median_genes=("n_genes_by_counts", "median"),
            median_pct_mt=("pct_counts_mt", "median"),
            q95_pct_mt=("pct_counts_mt", lambda x: x.quantile(0.95)),
        )
        .reset_index()
    )

    display(sample_qc_summary)

    sample_qc_summary.to_csv(
        RESULTS_DIR / "GSE184950_sample_QC_before_filtering.csv",
        index=False,
    )

In [18]:
# 17. QC distributions by sample
if adata is not None:
    samples = list(adata.obs["geo_accession"].astype(str).unique())

    # Counts
    fig, ax = plt.subplots(figsize=(12, 5))
    data = [
        np.log10(
            1 + adata.obs.loc[
                adata.obs["geo_accession"].astype(str) == s,
                "total_counts"
            ].values
        )
        for s in samples
    ]
    ax.boxplot(data, labels=samples, showfliers=False)
    ax.set_ylabel("log10(total counts + 1)")
    ax.set_xlabel("GEO sample")
    ax.set_title("Per-sample UMI count distributions")
    ax.tick_params(axis="x", rotation=90)
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / "GSE184950_QC_total_counts_by_sample.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

    # Genes detected
    fig, ax = plt.subplots(figsize=(12, 5))
    data = [
        adata.obs.loc[
            adata.obs["geo_accession"].astype(str) == s,
            "n_genes_by_counts"
        ].values
        for s in samples
    ]
    ax.boxplot(data, labels=samples, showfliers=False)
    ax.set_ylabel("Detected genes per nucleus")
    ax.set_xlabel("GEO sample")
    ax.set_title("Per-sample detected-gene distributions")
    ax.tick_params(axis="x", rotation=90)
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / "GSE184950_QC_genes_by_sample.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

# Part E — Data-driven QC thresholds

The purpose is to remove clear technical extremes without over-filtering vulnerable neuronal populations.

For each sample:

- lower detected-gene threshold:
  - at least 200 genes;
  - robust lower bound based on the sample distribution.

- upper detected-gene threshold:
  - sample Q3 + 3×IQR, useful for potential multiplets/high-complexity extremes.

- upper UMI threshold:
  - sample Q3 + 3×IQR.

- mitochondrial threshold:
  - sample Q3 + 3×IQR;
  - capped at 20% as a broad safety ceiling.

Thresholds are saved per sample and can be manually reviewed before final filtering.

In [19]:
# 18. Derive sample-aware QC thresholds
def robust_upper(series, cap=None):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    val = q3 + 3 * iqr
    return min(val, cap) if cap is not None else val

def robust_lower(series, floor=None):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    val = q1 - 3 * iqr
    return max(val, floor) if floor is not None else val

qc_threshold_rows = []

if adata is not None:
    for gsm, df in adata.obs.groupby("geo_accession", observed=True):
        qc_threshold_rows.append({
            "geo_accession": gsm,
            "min_genes": max(
                200,
                robust_lower(df["n_genes_by_counts"], floor=200)
            ),
            "max_genes": robust_upper(df["n_genes_by_counts"]),
            "max_counts": robust_upper(df["total_counts"]),
            "max_pct_mt": robust_upper(df["pct_counts_mt"], cap=20.0),
            "nuclei_before": len(df),
        })

qc_thresholds = pd.DataFrame(qc_threshold_rows)

if not qc_thresholds.empty:
    display(qc_thresholds)
    qc_thresholds.to_csv(
        RESULTS_DIR / "GSE184950_sample_specific_QC_thresholds.csv",
        index=False,
    )

In [20]:
# 19. Apply QC filters
if adata is not None and not qc_thresholds.empty:
    thresholds = qc_thresholds.set_index("geo_accession")

    keep = np.ones(adata.n_obs, dtype=bool)

    for gsm, idx in adata.obs.groupby(
        "geo_accession", observed=True
    ).groups.items():
        idx = list(idx)
        obs = adata.obs.loc[idx]
        t = thresholds.loc[gsm]

        local_keep = (
            (obs["n_genes_by_counts"] >= t["min_genes"]) &
            (obs["n_genes_by_counts"] <= t["max_genes"]) &
            (obs["total_counts"] <= t["max_counts"]) &
            (obs["pct_counts_mt"] <= t["max_pct_mt"])
        )

        pos = adata.obs_names.get_indexer(idx)
        keep[pos] = local_keep.values

    adata.obs["pass_basic_qc"] = keep

    print("Nuclei before QC:", adata.n_obs)
    print("Nuclei passing QC:", int(keep.sum()))
    print("Fraction retained:", keep.mean())

    qc_retention = (
        adata.obs.groupby(
            ["geo_accession", "disease"], observed=True
        )["pass_basic_qc"]
        .agg(["count", "sum", "mean"])
        .rename(columns={
            "count": "nuclei_before",
            "sum": "nuclei_after",
            "mean": "retained_fraction",
        })
        .reset_index()
    )

    display(qc_retention)
    qc_retention.to_csv(
        RESULTS_DIR / "GSE184950_QC_retention_by_sample.csv",
        index=False,
    )

    adata_qc = adata[adata.obs["pass_basic_qc"]].copy()
else:
    adata_qc = None

## Optional doublet detection

Doublets can distort clustering and apparent hybrid cell identities.

If `scrublet` is installed, this notebook calculates a doublet score **separately by GEO sample**.  
Because expected doublet rates depend on loading density and study-specific details, predicted doublets are stored as an annotation and not automatically removed without review.

In [21]:
# 20. Optional Scrublet per sample
if adata_qc is not None and HAS_SCRUBLET:
    import scrublet as scr
    from scipy import sparse

    adata_qc.obs["doublet_score"] = np.nan
    adata_qc.obs["predicted_doublet"] = False

    for gsm in adata_qc.obs["geo_accession"].astype(str).unique():
        mask = adata_qc.obs["geo_accession"].astype(str) == gsm
        sub = adata_qc[mask]

        counts = sub.layers["counts"]

        try:
            scrub = scr.Scrublet(
                counts,
                random_state=RANDOM_STATE,
            )
            scores, predictions = scrub.scrub_doublets()

            adata_qc.obs.loc[mask, "doublet_score"] = scores
            adata_qc.obs.loc[mask, "predicted_doublet"] = predictions

        except Exception as exc:
            warnings.warn(f"Scrublet failed for {gsm}: {exc}")

    display(
        adata_qc.obs.groupby(
            "geo_accession", observed=True
        )["predicted_doublet"]
        .agg(["count", "sum", "mean"])
    )
elif adata_qc is not None:
    print(
        "Scrublet not installed; doublet annotation skipped. "
        "Install `scrublet` to enable this optional step."
    )

# Part F — Normalization for visualization and dimensionality reduction

From this point onward:

- `adata_qc.layers["counts"]` remains raw counts.
- `adata_qc.X` becomes library-size normalized and log1p transformed.
- This normalized matrix is **not** used later as input to PyDESeq2 pseudobulk.

Workflow:
1. normalize total counts;
2. log1p;
3. highly variable genes;
4. scale HVGs;
5. PCA;
6. neighbors;
7. UMAP;
8. Leiden clustering.

In [22]:
# 21. Normalize and log-transform
if adata_qc is not None:
    # Preserve pre-normalization values in counts layer already.
    sc.pp.normalize_total(
        adata_qc,
        target_sum=1e4,
    )
    sc.pp.log1p(adata_qc)

    # Save normalized/log-transformed full matrix snapshot.
    adata_qc.raw = adata_qc

    print("Normalization completed.")

In [23]:
# 22. Highly variable genes
if adata_qc is not None:
    sc.pp.highly_variable_genes(
        adata_qc,
        n_top_genes=min(3000, adata_qc.n_vars),
        flavor="seurat",
        subset=False,
    )

    print(
        "Highly variable genes:",
        int(adata_qc.var["highly_variable"].sum())
    )

    hvg_table = (
        adata_qc.var.loc[
            adata_qc.var["highly_variable"],
            ["means", "dispersions", "dispersions_norm"]
        ]
        .sort_values("dispersions_norm", ascending=False)
    )

    hvg_table.to_csv(
        RESULTS_DIR / "GSE184950_highly_variable_genes.csv"
    )

In [24]:
# 23. PCA / neighbors / UMAP / Leiden
if adata_qc is not None:
    adata_embed = adata_qc[:, adata_qc.var["highly_variable"]].copy()

    sc.pp.scale(
        adata_embed,
        max_value=10,
    )

    sc.tl.pca(
        adata_embed,
        svd_solver="arpack",
        random_state=RANDOM_STATE,
    )

    n_pcs = min(30, adata_embed.obsm["X_pca"].shape[1])

    sc.pp.neighbors(
        adata_embed,
        n_neighbors=15,
        n_pcs=n_pcs,
        random_state=RANDOM_STATE,
    )

    sc.tl.umap(
        adata_embed,
        random_state=RANDOM_STATE,
    )

    try:
        sc.tl.leiden(
            adata_embed,
            resolution=0.6,
            key_added="leiden",
            random_state=RANDOM_STATE,
        )
    except Exception as exc:
        warnings.warn(
            "Leiden clustering was not available. "
            "Install `igraph` and `leidenalg` if needed. "
            f"Details: {exc}"
        )
        adata_embed.obs["leiden"] = "not_computed"

    # Transfer embeddings/clusters back into full-gene object.
    adata_qc.obsm["X_pca"] = adata_embed.obsm["X_pca"]
    adata_qc.obsm["X_umap"] = adata_embed.obsm["X_umap"]
    adata_qc.obs["leiden"] = adata_embed.obs["leiden"].astype(str).values

    print("Embedding completed.")

In [25]:
# 24. PCA variance
if adata_qc is not None and "pca" in adata_embed.uns:
    variance_ratio = adata_embed.uns["pca"]["variance_ratio"]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(
        np.arange(1, min(40, len(variance_ratio)) + 1),
        variance_ratio[:40],
        marker="o",
        markersize=3,
    )
    ax.set_xlabel("Principal component")
    ax.set_ylabel("Explained variance ratio")
    ax.set_title("GSE184950 PCA variance")
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / "GSE184950_PCA_variance.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

In [26]:
# 25. UMAP plots
if adata_qc is not None and "X_umap" in adata_qc.obsm:
    umap = adata_qc.obsm["X_umap"]

    for variable in [
        "disease",
        "geo_accession",
        "donor_id",
        "sex",
        "leiden",
    ]:
        if variable not in adata_qc.obs.columns:
            continue

        cats = adata_qc.obs[variable].astype("category")
        codes = cats.cat.codes.values

        fig, ax = plt.subplots(figsize=(7, 6))
        scatter = ax.scatter(
            umap[:, 0],
            umap[:, 1],
            c=codes,
            s=2,
            alpha=0.65,
        )
        ax.set_xlabel("UMAP1")
        ax.set_ylabel("UMAP2")
        ax.set_title(f"GSE184950 UMAP — {variable}")

        # Text legend outside the plot
        handles = []
        labels = []
        for i, cat in enumerate(cats.cat.categories):
            h = ax.scatter([], [], c=[i], s=15)
            handles.append(h)
            labels.append(str(cat))

        if len(labels) <= 40:
            ax.legend(
                handles,
                labels,
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                fontsize=7,
                markerscale=1,
            )

        fig.tight_layout()
        fig.savefig(
            FIGURES_DIR / f"GSE184950_UMAP_{variable}.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()

# Part G — Preliminary cell-type marker scoring

These scores are **exploratory annotations**, not final published labels.

Canonical marker groups are used only to:
- inspect cluster identity;
- detect obvious mixed/low-quality clusters;
- prepare for verification using study-supplied annotation.

Major expected substantia nigra populations include:
- dopaminergic neurons;
- other neurons;
- astrocytes;
- oligodendrocytes;
- OPCs;
- microglia;
- endothelial cells;
- pericytes.

No final disease inference is made from these scores.

In [27]:
# 26. Canonical marker sets
MARKER_SETS = {
    "Dopaminergic_neuron": [
        "TH", "SLC6A3", "DDC", "SLC18A2", "NR4A2", "PBX1"
    ],
    "Neuron": [
        "RBFOX3", "SNAP25", "SYT1", "MAP2", "NEFL"
    ],
    "Astrocyte": [
        "AQP4", "GFAP", "SLC1A3", "ALDH1L1", "S100B"
    ],
    "Oligodendrocyte": [
        "MBP", "MOG", "PLP1", "MAG", "MOBP"
    ],
    "OPC": [
        "PDGFRA", "CSPG4", "OLIG1", "OLIG2"
    ],
    "Microglia": [
        "P2RY12", "CX3CR1", "TMEM119", "AIF1", "C1QA"
    ],
    "Endothelial": [
        "CLDN5", "FLT1", "KDR", "EMCN", "PECAM1"
    ],
    "Pericyte": [
        "RGS5", "PDGFRB", "CSPG4", "COL4A1"
    ],
}

In [28]:
# 27. Score marker sets
marker_availability = []

if adata_qc is not None:
    for label, genes in MARKER_SETS.items():
        available = [g for g in genes if g in adata_qc.var_names]

        marker_availability.append({
            "marker_set": label,
            "requested_markers": len(genes),
            "available_markers": len(available),
            "genes_available": ";".join(available),
        })

        if len(available) >= 2:
            sc.tl.score_genes(
                adata_qc,
                gene_list=available,
                score_name=f"score_{label}",
                random_state=RANDOM_STATE,
            )

marker_availability = pd.DataFrame(marker_availability)

if not marker_availability.empty:
    display(marker_availability)
    marker_availability.to_csv(
        RESULTS_DIR / "GSE184950_marker_availability.csv",
        index=False,
    )

In [29]:
# 28. Cluster-level marker-score summary
cluster_marker_scores = pd.DataFrame()

if adata_qc is not None:
    score_cols = [
        c for c in adata_qc.obs.columns if c.startswith("score_")
    ]

    if score_cols:
        cluster_marker_scores = (
            adata_qc.obs
            .groupby("leiden", observed=True)[score_cols]
            .mean()
        )

        display(cluster_marker_scores)

        cluster_marker_scores.to_csv(
            RESULTS_DIR / "GSE184950_cluster_marker_scores.csv"
        )

In [30]:
# 29. Provisional marker-based cluster labels
cluster_label_map = {}

if not cluster_marker_scores.empty:
    for cluster, row in cluster_marker_scores.iterrows():
        best_score_col = row.idxmax()
        label = best_score_col.replace("score_", "")
        cluster_label_map[str(cluster)] = label

    adata_qc.obs["cell_type_provisional"] = (
        adata_qc.obs["leiden"]
        .astype(str)
        .map(cluster_label_map)
        .fillna("Unassigned")
    )

    print("Provisional cluster label map:")
    display(pd.Series(cluster_label_map, name="provisional_label"))

    print(
        "\nIMPORTANT: these labels are marker-based and must be "
        "verified against published annotation/marker expression."
    )

# Part H — Composition QC

Cell-type proportions are descriptive here.

Because each donor contributes many nuclei, naïvely comparing nucleus counts between disease groups would cause pseudoreplication.

Therefore composition summaries are exported by:
- sample;
- donor;
- disease;
- provisional cell type.

Formal composition testing, if desired, requires donor-level methods and is outside this preprocessing notebook.

In [31]:
# 30. Sample/donor composition tables
if adata_qc is not None:
    celltype_col = (
        "cell_type_provisional"
        if "cell_type_provisional" in adata_qc.obs.columns
        else "leiden"
    )

    sample_composition = (
        adata_qc.obs
        .groupby(
            [
                "geo_accession", "donor_id",
                "disease", celltype_col
            ],
            observed=True
        )
        .size()
        .rename("n_nuclei")
        .reset_index()
    )

    sample_totals = (
        sample_composition
        .groupby("geo_accession")["n_nuclei"]
        .transform("sum")
    )
    sample_composition["fraction_within_sample"] = (
        sample_composition["n_nuclei"] / sample_totals
    )

    donor_composition = (
        sample_composition
        .groupby(
            ["donor_id", "disease", celltype_col],
            observed=True
        )["n_nuclei"]
        .sum()
        .reset_index()
    )

    donor_totals = (
        donor_composition
        .groupby("donor_id")["n_nuclei"]
        .transform("sum")
    )
    donor_composition["fraction_within_donor"] = (
        donor_composition["n_nuclei"] / donor_totals
    )

    sample_composition.to_csv(
        RESULTS_DIR / "GSE184950_cell_composition_by_sample.csv",
        index=False,
    )
    donor_composition.to_csv(
        RESULTS_DIR / "GSE184950_cell_composition_by_donor.csv",
        index=False,
    )

    display(donor_composition.head(20))

# Part I — Prepare raw counts for Notebook 07 pseudobulk

Notebook 07 will aggregate:

> raw counts → donor × cell type → summed pseudobulk counts

Important:
- technical/sample rows sharing the same donor are merged at donor level;
- raw counts, not log-normalized values, are summed;
- minimum nuclei per donor × cell type will be checked before inferential modeling;
- Control vs PD is primary;
- PDD is retained for secondary analyses.

This notebook does not perform the pseudobulk differential-expression test yet.

In [32]:
# 31. Pseudobulk readiness table
pseudobulk_readiness = pd.DataFrame()

if adata_qc is not None:
    celltype_col = (
        "cell_type_provisional"
        if "cell_type_provisional" in adata_qc.obs.columns
        else "leiden"
    )

    pseudobulk_readiness = (
        adata_qc.obs
        .groupby(
            ["donor_id", "disease", celltype_col],
            observed=True
        )
        .size()
        .rename("n_nuclei")
        .reset_index()
    )

    pseudobulk_readiness["enough_nuclei_20"] = (
        pseudobulk_readiness["n_nuclei"] >= 20
    )

    display(
        pseudobulk_readiness
        .groupby(["disease", celltype_col], observed=True)
        .agg(
            donors=("donor_id", "nunique"),
            donors_ge20_nuclei=("enough_nuclei_20", "sum"),
            median_nuclei=("n_nuclei", "median"),
        )
        .reset_index()
    )

    pseudobulk_readiness.to_csv(
        RESULTS_DIR / "GSE184950_pseudobulk_readiness.csv",
        index=False,
    )

In [33]:
# 32. Save processed AnnData
if adata_qc is not None:
    output_h5ad = PROCESSED_DIR / "GSE184950_preprocessed_QC.h5ad"

    # Ensure raw counts are present.
    assert "counts" in adata_qc.layers

    adata_qc.write_h5ad(
        output_h5ad,
        compression="gzip",
    )

    print("Saved:", output_h5ad.resolve())
else:
    print(
        "AnnData not saved because expression matrices were not available."
    )

AnnData not saved because expression matrices were not available.


# Part J — Reproducibility manifest and checkpoint

In [34]:
# 33. Manifest
manifest06 = {
    "dataset": "GSE184950",
    "modality": "single-nucleus RNA-seq",
    "tissue": "substantia nigra",
    "series_matrix_found": SERIES_FILE is not None,
    "raw_tar_found": RAW_TAR is not None,
    "add2_annotation_found": ADD2_FILE is not None,
    "valid_10x_directories": int(len(tenx_table)),
    "expression_available": bool(EXPRESSION_AVAILABLE),
    "scanpy_available": bool(HAS_SCANPY),
    "anndata_available": bool(HAS_ANNDATA),
    "expression_processed": bool(adata_qc is not None),
    "sample_rows": int(len(meta)),
    "unique_donors": int(meta["donor_id"].nunique()),
    "primary_validation_contrast": "PD vs Control",
    "secondary_group": "PDD",
    "biological_replicate": "donor",
    "raw_count_layer": "counts",
    "cell_labels": (
        "provisional marker-based unless published annotation is supplied"
    ),
    "next_notebook": "07_cell_type_PD_analysis.ipynb",
}

with open(
    PROCESSED_DIR / "notebook06_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest06, f, indent=2, ensure_ascii=False)

print(json.dumps(manifest06, indent=2, ensure_ascii=False))

{
  "dataset": "GSE184950",
  "modality": "single-nucleus RNA-seq",
  "tissue": "substantia nigra",
  "series_matrix_found": true,
  "raw_tar_found": false,
  "add2_annotation_found": false,
  "valid_10x_directories": 0,
  "expression_available": false,
  "scanpy_available": false,
  "anndata_available": false,
  "expression_processed": false,
  "sample_rows": 34,
  "unique_donors": 32,
  "primary_validation_contrast": "PD vs Control",
  "secondary_group": "PDD",
  "biological_replicate": "donor",
  "raw_count_layer": "counts",
  "cell_labels": "provisional marker-based unless published annotation is supplied",
  "next_notebook": "07_cell_type_PD_analysis.ipynb"
}


In [35]:
# 34. Final checkpoint
checks = {
    "GEO metadata loaded": len(meta) == 34,
    "Unique GSM IDs": meta["geo_accession"].nunique() == 34,
    "Donor IDs available": "donor_id" in meta.columns,
    "Disease labels available": set(["Control", "PD"]).issubset(
        set(meta["disease"].unique())
    ),
    "Repeated donors audited": True,
}

if adata_qc is not None:
    checks.update({
        "Raw counts preserved": "counts" in adata_qc.layers,
        "Cell IDs unique": adata_qc.obs_names.is_unique,
        "Donor metadata attached": "donor_id" in adata_qc.obs.columns,
        "Disease metadata attached": "disease" in adata_qc.obs.columns,
        "UMAP available": "X_umap" in adata_qc.obsm,
        "Pseudobulk readiness exported": len(pseudobulk_readiness) > 0,
    })

checkpoint = pd.Series(checks, name="passed")
display(checkpoint)

assert checkpoint.all(), "Notebook 06 checkpoint failed."

if adata_qc is not None:
    print("\n✓ Notebook 06 completed with expression preprocessing.")
    print(
        "The processed AnnData is ready for donor × cell-type "
        "pseudobulk construction in Notebook 07."
    )
else:
    print("\n✓ Notebook 06 metadata/QC framework completed.")
    print(
        "⚠ Expression preprocessing is pending because complete 10x "
        "matrices and/or Scanpy were not available in this runtime."
    )
    print(
        "Provide GSE184950_RAW.tar or extracted matrix.mtx + "
        "barcodes.tsv + features/genes.tsv files, install Scanpy, "
        "and rerun from Part B."
    )

GEO metadata loaded         True
Unique GSM IDs              True
Donor IDs available         True
Disease labels available    True
Repeated donors audited     True
Name: passed, dtype: bool


✓ Notebook 06 metadata/QC framework completed.
⚠ Expression preprocessing is pending because complete 10x matrices and/or Scanpy were not available in this runtime.
Provide GSE184950_RAW.tar or extracted matrix.mtx + barcodes.tsv + features/genes.tsv files, install Scanpy, and rerun from Part B.


# Interpretation and limitations

### What Notebook 06 establishes
- correct sample and donor identities;
- disease-group metadata;
- sample-specific technical QC;
- a donor-aware single-nucleus object;
- raw-count preservation for pseudobulk;
- exploratory dimensionality reduction and clustering;
- provisional cell-type structure.

### What it does not establish
- statistically significant PD-vs-Control differential expression;
- validated cell-type-specific metabolic signatures;
- causal metabolic dysfunction;
- independent validation of the GSE136666 signature.

Those are addressed in the next notebooks.

## Next step — Notebook 07

**Notebook 07 — Cell-Type PD Analysis**

For each sufficiently represented cell type:

1. aggregate raw counts by donor;
2. merge repeated sample rows belonging to the same donor;
3. evaluate donor and nucleus representation;
4. primary contrast: PD vs Control;
5. optional covariates only after balance/collinearity checks;
6. run PyDESeq2 pseudobulk differential expression;
7. evaluate the Notebook 05 metabolic signature;
8. retain PDD as a secondary/progression analysis.